In [ ]:
# this file looks at the local environment of each IF-annotated cell such as gene expression (predictors) and proportion dying (response).
# I used raw counts from the Imaris data and the previous file with spatial alignment of Visium bins and transcriptional data. 

# read in different dataframes depending on # of groups or individual genes of interest.

In [ ]:
# importing necessary libraries
import pandas as pd
import numpy as np
from tqdm import tqdm # for the progress bar
import os

In [ ]:
# setting my own working directory
os.chdir("i:/Hu Lab/Sophie/1. Cell death/visium image manual spot selection/20260413_final_merge/data")

In [ ]:
# V = pd.read_csv("iso7spatial_sig.csv")            # visium, known signatures
# V = pd.read_feather("iso7spatial__genes.feather") # visium, all genes
V = pd.read_csv("pd1spatial_sig.csv")

# S = pd.read_csv("iso7_coords_clean.csv") # spots from IF
S = pd.read_csv("pd1-9_coords_final.csv")  # spots from IF

In [ ]:
assert {"x", "y"}.issubset(V.columns)
assert {"x", "y", "cell_type"}.issubset(S.columns)  # sanity check

In [ ]:
gene_cols = V.columns[4:16]  # look at exact indices for category columns (not x,y).
print(gene_cols[:20])

In [ ]:
# parameters/ initializing things

In [ ]:
v_coords = V[["x", "y"]].to_numpy()
s_coords = S[["x", "y"]].dropna().to_numpy()

# maybe do NaN QC if data not clean

In [7]:
from scipy.spatial import cKDTree

In [ ]:
# Running the KNN tree to find neighbors

In [ ]:
def inputs(S, V, gene_cols, s_coords, v_coords):

    # KD-trees
    s_tree = cKDTree(s_coords)
    v_tree = cKDTree(v_coords)

    # Cell types as integers --> faster processing
    type_map = {
        "tdtomato": 0,
        "gc3ai": 1,
        "cd8": 2,
        "lectin": 3
    }
    S_cells = np.array([type_map.get(x, -1) for x in S["cell_type"].values])

    # extracting gene matrix :)
    V_genes = V[gene_cols].to_numpy()

    return s_tree, v_tree, S_cells, V_genes

In [ ]:
def counts_in_radius(center, s_tree, S_cells, radius):

    idx = s_tree.query_ball_point(center, r=radius)

    if len(idx) == 0:
        counts = np.zeros(4)   # for all four types, if nothing then set 0. Loops through all neighborhoods
    else:
        types = S_cells[idx]
        counts = np.bincount(types[types >= 0], minlength=4) # so our result: counts = [n_tdtomato, n_gc3ai, n_cd8, n_lectin]

    n_alive, n_dying, n_immune, n_endothelial = counts       # renaming the channels to what cell type they represent

    # calculating later metrics so I don't have to do it downstream
    tumor = n_alive + n_dying
    total = tumor + n_immune + n_endothelial
    prop_dying = n_dying / tumor if tumor > 0 else np.nan
    exist_dying = 1 if prop_dying > 0 else 0                        # binarizing proportion dying
    efficacy = prop_dying/ n_immune if n_immune > 0 else np.nan     # roportion dying per cd8, are some T-cells better at killing?

    return counts, tumor, total, prop_dying, exist_dying, efficacy

In [ ]:
radius = 50 # what I ended up choosing to balance capturing enough cells but avoiding violating independence too much. 
            # Ideal was 10 microns but too sparse. each cell ~ 8 microns exist overlap & study simplifies to 2D ignoring z-axis.

In [ ]:
def get_gene_means(center, v_tree, V_genes, radius): # means have greater smoothing & 
                                                     # in our idea better representation of gene expression than sums.

    idx = v_tree.query_ball_point(center, r=radius)

    if len(idx) == 0:
        return np.zeros(V_genes.shape[1])

    return V_genes[idx].mean(axis=0)

In [12]:
def append_row(center, s_tree, v_tree, S_types, V_genes, radius):

    counts, tumor, total, prop_dying, exist_dying, efficacy = counts_in_radius(
        center, s_tree, S_types, radius
    )

    gene_means = get_gene_means(
        center, v_tree, V_genes, radius
    )

    row = np.concatenate([
        np.array([center[0], center[1]]),
        counts,
        np.array([tumor, total, prop_dying, exist_dying, efficacy]),
        gene_means
    ])

    return row

In [ ]:
# our final assembly.
def compute_neighborhoods(
    S, V, s_coords, v_coords, gene_cols, radius):

    s_tree, v_tree, S_types, V_genes = inputs(
        S, V, gene_cols, s_coords, v_coords
    )

    n_centers = len(s_coords)
    n_genes = V_genes.shape[1]

    results = np.zeros((n_centers, 11 + n_genes))

    for i, center in enumerate(tqdm(s_coords, desc="Processing")):
        results[i] = append_row(
            center, s_tree, v_tree, S_types, V_genes, radius
        )

    columns = (
        ["cx", "cy",
         "n_alive", "n_dying", "n_immune", "n_lectin",
         "tumor", "all", "prop_dying", "exist_dying", "efficacy"]
        + list(gene_cols)
    )

    return pd.DataFrame(results, columns=columns)

In [ ]:
# actually running the function now. depending on how many genes or signatures, after the progress bar ends, 
# it may still take a while. just a heads up, sorry

df = compute_neighborhoods(
    S=S,
    V=V,
    s_coords=s_coords,
    v_coords=v_coords,
    gene_cols=gene_cols,
    radius=radius
)

In [ ]:
# cleaning a bit. Set tumor n = 30 because of CLT. 
# neighborhoods overlap spatially so not fully independent but capturing too few cells --> high variance

df = df[df["tumor"] > 30]
df = df[df["n_immune"] > 0] # removing for NaN efficacy denominator

In [ ]:
# verify output
print(df.iloc[3000:3005, :20])

In [ ]:
# df.to_feather("spatial_means_all.feather")    # feather is for all genes, too large a file
df.to_csv("iso7spatialSig_means.csv", index=False) 